In [ ]:
# =====================================================================
# PLANTILLA GENÉRICA: Clasificación Multiclase con Imágenes de Drive
# =====================================================================

# ---------------------------------------------------------------------
# PASO 1: Conectar a Google Drive
# ---------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------
# PASO 2: Importar Librerías
# ---------------------------------------------------------------------
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Selección automática de hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Trabajando con el dispositivo: {device}\n")

# ---------------------------------------------------------------------
# PASO 3: Configurar Rutas y Número de Clases (¡MODIFICA AQUÍ!)
# ---------------------------------------------------------------------
# Ajusta el nombre de la carpeta raíz si es necesario
RUTA_TRAIN = '/content/drive/MyDrive/EcoVisionAI/TRAIN/'
RUTA_TEST = '/content/drive/MyDrive/EcoVisionAI/TEST/'

# ---------------------------------------------------------------------
# PASO 4: Carga de Datos y Auto-detección de Clases
# ---------------------------------------------------------------------
transformaciones = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=RUTA_TRAIN, transform=transformaciones)
test_dataset = ImageFolder(root=RUTA_TEST, transform=transformaciones)

# Guardamos dinámicamente cuántas clases detectó ImageFolder en Drive
NUM_CLASES = len(train_dataset.classes)

print(f"¡Configuración Multiclase Detectada!")
print(f"Número de clases encontradas: {NUM_CLASES}")
print(f"Nombres de las clases: {train_dataset.classes}")
print(f"Mapeo de índices: {train_dataset.class_to_idx}\n")

BATCH_SIZE = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------------
# PASO 5: Arquitectura ResNet18 con Transfer Learning
# ---------------------------------------------------------------------

import torchvision.models as models

# Cargar ResNet18 preentrenada en ImageNet
pesos = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=pesos)

# Obtener el número de características de la última capa
num_features = model.fc.in_features

# Reemplazar la última capa para adaptarla al número de clases
model.fc = nn.Linear(num_features, NUM_CLASES)

# Enviar el modelo a CPU o GPU
model = model.to(device)

# ---------------------------------------------------------------------
# PASO 6: Función de Pérdida y Optimidad
# ---------------------------------------------------------------------
# Nota para la clase: CrossEntropyLoss maneja internamente Softmax, por lo que
# es perfectamente compatible tanto para biclase como para multiclase en PyTorch.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=0.0001)

# ---------------------------------------------------------------------
# PASO 7: Bucle de Entrenamiento
# ---------------------------------------------------------------------
NUM_EPOCHS = 12
print("Iniciando el entrenamiento multiclase...\n")

for epoch in range(NUM_EPOCHS):

    # ============================
    # ENTRENAMIENTO
    # ============================
    model.train()

    loss_acumulada = 0.0
    correctos_train = 0
    total_train = 0

    for imagenes, etiquetas in train_loader:

        imagenes = imagenes.to(device)
        etiquetas = etiquetas.to(device)

        outputs = model(imagenes)
        loss = criterion(outputs, etiquetas)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_acumulada += loss.item()

        _, predicciones = torch.max(outputs, 1)

        total_train += etiquetas.size(0)
        correctos_train += (predicciones == etiquetas).sum().item()

    loss_train = loss_acumulada / len(train_loader)
    accuracy_train = 100 * correctos_train / total_train

    # ============================
    # EVALUACIÓN EN TEST
    # ============================
    model.eval()

    loss_test = 0.0
    correctos_test = 0
    total_test = 0

    with torch.no_grad():

        for imagenes, etiquetas in test_loader:

            imagenes = imagenes.to(device)
            etiquetas = etiquetas.to(device)

            outputs = model(imagenes)

            loss = criterion(outputs, etiquetas)
            loss_test += loss.item()

            _, predicciones = torch.max(outputs, 1)

            total_test += etiquetas.size(0)
            correctos_test += (predicciones == etiquetas).sum().item()

    loss_test = loss_test / len(test_loader)
    accuracy_test = 100 * correctos_test / total_test

    # ============================
    # RESULTADOS DE LA ÉPOCA
    # ============================
    print(f"Época [{epoch+1}/{NUM_EPOCHS}] / Pérdida Train: {loss_train:.4f} / Train Accuracy: {accuracy_train:.2f}% / Perdida Test: {loss_test:.4f} / Test Accuracy: {accuracy_test:.2f}%")
    print("-" * 45)

print("\n¡Entrenamiento concluido!")
# ---------------------------------------------------------------------
# PASO 8: Evaluación Final en el Set de Prueba (Test)
# ---------------------------------------------------------------------
model.eval()
correctos = 0
total_imagenes = 0

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        outputs = model(imagenes)
        _, predicciones_indices = torch.max(outputs.data, 1)

        total_imagenes += etiquetas.size(0)
        correctos += (predicciones_indices == etiquetas).sum().item()

accuracy = 100 * correctos / total_imagenes
print(f"Exactitud (Accuracy) final en TEST para las {NUM_CLASES} clases: {accuracy:.2f}%")

torch.save(model.state_dict(), "/content/drive/MyDrive/EcoVisionAI/Modelos/EcoVisionAI_ResNet50.pth")
print("Modelo guardado correctamente")

Mounted at /content/drive
Trabajando con el dispositivo: cpu

¡Configuración Multiclase Detectada!
Número de clases encontradas: 10
Nombres de las clases: ['Basura', 'Bateria', 'Biologica', 'Carton', 'Metal', 'Papel', 'Plastico', 'Vidrio Cafe', 'Vidrio Verde', 'Vidrio blanco']
Mapeo de índices: {'Basura': 0, 'Bateria': 1, 'Biologica': 2, 'Carton': 3, 'Metal': 4, 'Papel': 5, 'Plastico': 6, 'Vidrio Cafe': 7, 'Vidrio Verde': 8, 'Vidrio blanco': 9}

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 151MB/s]


Iniciando el entrenamiento multiclase...



KeyboardInterrupt: 